[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_4_off_policy_control_complete.ipynb)

<div style="text-align:center">
    <h1>
        Off-policy Monte Carlo Control
    </h1>
</div>

<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_policy, plot_action_values


## Import the necessary software libraries:

In [ ]:
# NumPy gives us fast arrays for the Q-table and quick random choices.
import numpy as np
# Matplotlib lets us draw the maze, the Q-values and the learned policy.
import matplotlib.pyplot as plt

## Initialize the environment

In [ ]:
env = Maze()

In [ ]:
# Ask the environment for a picture (an RGB image) of the current maze.
frame = env.render(mode='rgb_array')
# Make the figure a comfortable size to look at.
plt.figure(figsize=(6, 6))
# Hide the x/y axis ticks so we just see the maze cleanly.
plt.axis('off')
# Display that image inside the notebook.
plt.imshow(frame)

In [ ]:
# Shape of the state space: how many rows and columns the grid has.
print(f"Observation space shape: {env.observation_space.nvec}")
# How many distinct actions the agent can take (up, down, left, right).
print(f"Number of actions: {env.action_space.n}")

## Define value table $Q(s, a)$

#### Create the $Q(s, a)$ table

In [ ]:
# Start every Q-value very low (-100) so unvisited actions look unattractive.
action_values = np.full((5,5,4), -100)
# The goal cell (4, 4) is terminal, so its value is exactly 0 (no future reward).
action_values[4,4,:] = 0.

#### Plot Q(s, a)

In [ ]:
# Draw the current Q-values so we can see them before any learning.
plot_action_values(action_values)

## Define the target policy $\pi(s)$

In off-policy learning we juggle **two policies at once**:

- the **target policy** $\pi(s)$ -- the policy we actually want to perfect. It is
  **purely greedy** (always takes the best-known action, no random exploration). This is
  the policy we will use at the end to solve the maze.
- the **behaviour policy** $b(s)$ (defined a bit further down) -- the policy that actually
  *drives the agent around* while learning. It explores, so it visits actions the greedy
  policy would never try on its own.

The whole point of off-policy methods: **explore with $b$, but learn about $\pi$.**

Below we create the greedy target policy.

#### Create the policy $\pi(s)$

In [ ]:
# The TARGET policy is the one we ultimately want: purely greedy, no exploration.
def target_policy(state):
    # Look up the four Q-values for this state.
    av = action_values[state]
    # Always pick the best action; break ties at random.
    return np.random.choice(np.flatnonzero(av == av.max()))

#### Test the policy with state (0, 0)

In [ ]:
# Ask the target (greedy) policy what it would do in state (0, 0).
action = target_policy((0,0))
# Print the chosen action so we can check it runs.
print(f"Action taken in state (0,0): {action}")

#### Plot the policy

In [ ]:
# Draw an arrow in each cell showing the greedy target policy's preferred action.
plot_policy(action_values, frame)

## Define the exploratory policy $b(s)$

This is the **behaviour policy** $b(s)$ -- the one that actually moves the agent during
training. It is **epsilon-greedy**: usually greedy, but with probability `epsilon` it
takes a random action. That random exploration is what lets the agent discover good paths
it would otherwise never sample.

Key idea: the agent **behaves** with this exploratory $b$, but it will **learn the values
of the greedy target policy** $\pi$ from that experience. Reweighting the experience so it
counts "as if" it came from $\pi$ is the job of **importance sampling**, used in the
algorithm below.

#### Create the policy $b(s)$

In [ ]:
# The BEHAVIOUR policy b(s): epsilon-greedy, so it actually explores the maze.
def exploratory_policy(state, epsilon=0.):
    # With probability epsilon, take a completely random action (explore).
    if np.random.random() < epsilon:
        return np.random.randint(4)
    else:
        # Otherwise look up the Q-values for this state.
        av = action_values[state]
        # Take the best action; break ties at random.
        return np.random.choice(np.flatnonzero(av == av.max()))

#### Test the policy with state (0, 0)

In [ ]:
# Ask the exploratory (behaviour) policy what it would do in state (0, 0).
action = exploratory_policy((0,0))
# Print the chosen action so we can check it runs.
print(f"Action taken in state (0,0): {action}")

#### Plot the policy

In [ ]:
# Draw the exploratory policy's current preferred action in each cell.
plot_policy(action_values, frame)

## Implement the algorithm

### On-policy vs off-policy -- the core difference

- **On-policy** (earlier notebooks): the agent learns about the **same** policy it uses
  to act. It must keep exploring forever, so it can only ever learn a *slightly random*
  policy.
- **Off-policy** (this notebook): the agent acts with an **exploratory** behaviour policy
  $b$, but learns the values of a **different, fully greedy** target policy $\pi$. This
  lets it explore freely while still learning the *optimal* (non-random) policy.

### The catch: the data comes from the "wrong" policy

The episodes are generated by $b$, but we want returns as if they came from $\pi$. Some
actions $b$ takes are ones $\pi$ would *never* choose. We correct for this mismatch with
**importance sampling**.

### Importance sampling, in plain words

Each step gets an **importance weight** `W`, which answers: *"how much more (or less)
likely was this exact action under the target policy $\pi$ than under the behaviour
policy $b$?"* We build it up as we walk backwards through the episode:

- If the behaviour action **matches** what the greedy target would do, the experience is
  relevant, so we keep it and **scale `W`** by `1 / b(action | state)` (rarer behaviour
  choices count for more).
- If the behaviour action **does not match** the greedy target's choice, the target policy
  would have $0$ probability of going that way, so everything *before* this point is
  irrelevant -- we **stop** (the `break`) and move to the next episode.

### Weighted importance sampling update

Instead of a plain average, we track the running total weight `csa` for each
`(state, action)` pair and update:

```
Q(s, a)  <-  Q(s, a)  +  (W / csa) * ( G - Q(s, a) )
```

So returns that are more "$\pi$-like" (larger `W`) pull the estimate harder. The result:
the agent explores with $b$, yet the `Q` table converges to the values of the **optimal
greedy policy** $\pi$ -- which is exactly the policy we test at the end.

In [ ]:
# Off-policy Monte Carlo control using weighted importance sampling.
def off_policy_mc_control(action_values, target_policy, exploratory_policy, episodes, gamma=0.99, epsilon=0.2):

    # Repeat the learn-from-one-episode process many times.
    for episode in range(1, episodes + 1):
        # G is the return (discounted future reward); reset to 0 each episode.
        G = 0
        # W is the importance-sampling weight; it starts at 1 for each episode.
        W = 1
        # csa accumulates the total weight given to each (state, action) pair this episode.
        csa = np.zeros((5, 5, 4))
        # Start a fresh episode and get the first state.
        state = env.reset()
        # 'done' becomes True when the agent reaches the goal.
        done = False
        # Record the trajectory (state, action, reward) of this episode.
        transitions = []

        # Generate the episode using the EXPLORATORY (behaviour) policy.
        while not done:
            # Act with the exploring policy b(s), not the greedy target policy.
            action = exploratory_policy(state, epsilon)
            # Take the action and observe the next state and reward.
            next_state, reward, done, _ = env.step(action)
            # Save the step for the backward learning pass.
            transitions.append([state, action, reward])
            # Move to the next state.
            state = next_state

        # Learn from the episode by walking backwards through it.
        for state_t, action_t, reward_t in reversed(transitions):
            # Update the return with this step's reward.
            G = reward_t + gamma * G
            # Add the current weight to this pair's running total weight.
            csa[state_t][action_t] += W
            # Current estimate of this (state, action) value.
            qsa = action_values[state_t][action_t]
            # Move Q toward G, scaled by this step's share of the total weight.
            action_values[state_t][action_t] += (W / csa[state_t][action_t]) * (G - qsa)

            # If the behaviour action differs from what the greedy target would do,
            # the trajectories diverge here, so stop updating earlier steps.
            if action_t != target_policy(state_t):
                break

            # Otherwise grow the importance weight by 1 / b(action_t | state_t).
            # b chooses the greedy action with probability (1 - epsilon + epsilon/4).
            W = W * 1. / (1 - epsilon + epsilon/4)

In [ ]:
# Train: run off-policy MC control for 1,000 episodes with epsilon = 0.3.
off_policy_mc_control(action_values, target_policy, exploratory_policy, episodes=1000, epsilon=0.3)

## Show results

#### Show resulting value table $Q(s, a)$

In [ ]:
# Plot the learned Q-values after training.
plot_action_values(action_values)

#### Show resulting policy $\pi(\cdot|s)$

In [ ]:
# Draw the final greedy target policy; arrows should now point toward the goal.
plot_policy(action_values, frame)

#### Test the resulting agent

In [ ]:
# Evaluate the GREEDY TARGET policy (the policy we were really learning about).
test_agent(env, target_policy)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch. 5: Monte Carlo methods](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)